# PBMC 1k v3: FASTQ → Cell Ranger → AnnData

This notebook runs **10x Cell Ranger `count`** on paired **R1/R2** FASTQs under `assets/pbmc_1k/pbmc_1k_v3_fastqs`, then loads **`outs/filtered_feature_bc_matrix`** into an **`AnnData`** object (cells × genes).

## Prerequisites

1. **Cell Ranger** installed and on your `PATH` (see [10x support](https://www.10xgenomics.com/support/software/cell-ranger)).
2. A **reference transcriptome** compatible with your Cell Ranger build (for this human demo, use a 10x-compatible **GRCh38** reference, e.g. `refdata-gex-GRCh38-2020-A` from 10x [downloads](https://www.10xgenomics.com/support/software/cell-ranger/downloads)).
3. Enough **CPU/RAM** for a full run (Cell Ranger documents typical requirements; adjust `--localcores` / `--localmem` below).

**`--sample`**: must match the FASTQ name prefix (here `pbmc_1k_v3` in `pbmc_1k_v3_S1_L00*_R*_001.fastq.gz`). Cell Ranger uses **R1/R2** together; index reads (e.g. **I1**) are ignored automatically.

Set `RUN_CELLRANGER = True` in the run cell when you are ready to execute **`cellranger count`** (it can take a long time). If `cellranger` is not on your `PATH`, set **`CELLRANGER_BIN`** to the full path of the executable. If the output directory already exists, either delete it or change `CR_RUN_ID`.

In [1]:
from __future__ import annotations

import os
import shutil
import subprocess
from pathlib import Path

import scanpy as sc

In [9]:
REPO_ROOT = Path("..").resolve()
FASTQ_DIR = REPO_ROOT / "assets" / "pbmc_1k" / "pbmc_1k_v3_fastqs"

# 10x reference unpacked from refdata-gex-*.tar.gz (must match your Cell Ranger version).
TRANSCRIPTOME = REPO_ROOT / "assets" / "plos_2016" / "refs" / "refdata-gex-GRCh38-2024-A.gz"
# Optional: full path to cellranger if not on PATH (e.g. /opt/cellranger-9.0.0/cellranger).
CELLRANGER_BIN = os.environ.get("CELLRANGER_BIN", "cellranger")

CR_RUN_ID = "pbmc_1k_v3_cellranger_count"
CR_OUT = REPO_ROOT / "assets" / "pbmc_1k" / CR_RUN_ID

OUT_H5AD = REPO_ROOT / "assets" / "pbmc_1k" / "pbmc_1k_v3_cellranger.h5ad"

LOCALCORES = min(8, os.cpu_count() or 8)
LOCALMEM = 12

sorted(FASTQ_DIR.glob("*.fastq.gz"))

[PosixPath('/Users/fabro/Documents/UBA/TESIS/repos/single-cell/assets/pbmc_1k/pbmc_1k_v3_fastqs/pbmc_1k_v3_S1_L001_I1_001.fastq.gz'),
 PosixPath('/Users/fabro/Documents/UBA/TESIS/repos/single-cell/assets/pbmc_1k/pbmc_1k_v3_fastqs/pbmc_1k_v3_S1_L001_R1_001.fastq.gz'),
 PosixPath('/Users/fabro/Documents/UBA/TESIS/repos/single-cell/assets/pbmc_1k/pbmc_1k_v3_fastqs/pbmc_1k_v3_S1_L001_R2_001.fastq.gz'),
 PosixPath('/Users/fabro/Documents/UBA/TESIS/repos/single-cell/assets/pbmc_1k/pbmc_1k_v3_fastqs/pbmc_1k_v3_S1_L002_I1_001.fastq.gz'),
 PosixPath('/Users/fabro/Documents/UBA/TESIS/repos/single-cell/assets/pbmc_1k/pbmc_1k_v3_fastqs/pbmc_1k_v3_S1_L002_R1_001.fastq.gz'),
 PosixPath('/Users/fabro/Documents/UBA/TESIS/repos/single-cell/assets/pbmc_1k/pbmc_1k_v3_fastqs/pbmc_1k_v3_S1_L002_R2_001.fastq.gz')]

In [10]:
def infer_10x_sample_prefix(fastq_dir: Path) -> str:
    """Prefix before '_S*_L*_R*' in 10x-style FASTQ names."""
    r1 = next(iter(sorted(fastq_dir.glob("*_R1_*.fastq.gz"))))
    stem = r1.name.replace(".fastq.gz", "")
    return stem.split("_S")[0]


SAMPLE = infer_10x_sample_prefix(FASTQ_DIR)
SAMPLE

'pbmc_1k_v3'

In [11]:
n = 1000

In [15]:
RUN_CELLRANGER = True

if RUN_CELLRANGER:
    """
    cand = Path(CELLRANGER_BIN).expanduser()
    if cand.is_file():
        cellranger_bin = str(cand.resolve())
    else:
        cellranger_bin = shutil.which(CELLRANGER_BIN) or shutil.which("cellranger")
    if cellranger_bin is None:
        raise FileNotFoundError(
            "cellranger not found. Install on PATH or set CELLRANGER_BIN to the executable path."
        )
    """
    tx = TRANSCRIPTOME
    if CR_OUT.exists():
        raise FileExistsError(f"Remove or rename existing output: {CR_OUT}")
    CR_OUT.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
    "docker", "run", "--rm",
    "-p", "3600:3600",
    "-v", f"{Path(FASTQ_DIR).resolve()}:/data/fastqs",
    "-v", f"{Path(TRANSCRIPTOME).resolve()}:/data/ref",
    "-v", f"{Path(CR_OUT.parent).resolve()}:/data/output",
    "cellranger",
    "count",
    "--uiport=3600",
    f"--id={CR_RUN_ID}",
    "--transcriptome=/data/ref",
    "--fastqs=/data/fastqs",
    f"--sample={SAMPLE}",
    f"--expect-cells={n}",
    f"--localcores={LOCALCORES}",
    f"--localmem={LOCALMEM}",
    "--create-bam=false",
]
    subprocess.run(cmd, cwd=str(CR_OUT.parent), check=True)
else:
    print("Set RUN_CELLRANGER = True to run cellranger count.")
    print("Expected:", CR_OUT / "outs" / "filtered_feature_bc_matrix.h5")

Thank you for using cellranger. To help us improve our product,
anonymized telemetry data has been collected and sent to 10X Genomics.
This data helps us understand usage patterns, diagnose issues,
and prioritize improvements.

You can inspect the telemetry metrics sent by looking in
/root/.cache/tenx/telemetry/cellranger/10.0.0

For more details on what data is collected and how it's used, please visit
https://10xgen.com/pipeline-telemetry

You can disable telemetry at any time by running the following command:
	cellranger telemetry disable




Martian Runtime - v4.0.14
2026-03-28 19:27:38 [jobmngr] WARNING: configured to use 12GB of local memory, but only 6.8GB is currently available.
Serving UI at http://fc3f59673dbd:3600?auth=30w57BV8Ifn06gA7tt1p4KWASPvui_r5DhE3dzBq5KI

Running preflight checks (please wait)...


KeyboardInterrupt: 

In [39]:
outs = CR_OUT / "outs"
h5 = outs / "filtered_feature_bc_matrix.h5"
mtx_dir = outs / "filtered_feature_bc_matrix"

if h5.is_file():
    adata = sc.read_10x_h5(h5, gex_only=True)
elif (mtx_dir / "matrix.mtx.gz").is_file() or (mtx_dir / "matrix.mtx").is_file():
    adata = sc.read_10x_mtx(mtx_dir, gex_only=True)
else:
    raise FileNotFoundError(
        f"No filtered matrix at {h5} or {mtx_dir}. Run Cell Ranger with RUN_CELLRANGER = True."
    )

adata.var_names_make_unique()
adata.obs_names_make_unique()
adata.obs["sample"] = SAMPLE
adata.uns["cellranger_run"] = str(CR_OUT)

adata

FileNotFoundError: No filtered matrix at /Users/fabro/Documents/UBA/TESIS/repos/single-cell/assets/pbmc_1k/pbmc_1k_v3_cellranger_count/outs/filtered_feature_bc_matrix.h5 or /Users/fabro/Documents/UBA/TESIS/repos/single-cell/assets/pbmc_1k/pbmc_1k_v3_cellranger_count/outs/filtered_feature_bc_matrix. Run Cell Ranger with RUN_CELLRANGER = True.

In [ ]:
adata.write_h5ad(OUT_H5AD)
OUT_H5AD